# Inspired Final Model v2: 정교한 스무딩 및 운영 심리 반영

이 노트북은 v1의 'Weak Correction'을 한 단계 더 발전시킨 버전입니다.
**핵심 추가 전략:**
1. **메뉴 보정 스무딩 (M-estimator):** 메뉴 출현 횟수가 적으면 보정값을 0에 가깝게, 많으면 확실하게 반영하여 과적합을 방지합니다.
2. **월급날 및 기간 효과:** 25일(월급날) 효과와 월말 효과를 정교하게 계산합니다.
3. **석식 특수일 보정:** 수요일/금요일 시그널을 연휴 정보와 결합하여 더 정밀하게 조정합니다.
4. **앙상블 대신 단일 모델 내실화:** 앙상블의 노이즈를 피하고 베이스 모델의 안정성을 극대화합니다.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색 (CY2 환경)
def find_path(cands): 
    for p in cands: 
        if os.path.exists(p): return p
    return None

train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\train_median.csv", r"./data/train_median.csv"])
test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\test.csv", r"./data/test.csv"])
sub_path = find_path([r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv", r"./data/sample_submission.csv"])
w_train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weather.csv", r"./data/weather.csv"])
w_test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weatherTest.csv", r"./data/weatherTest.csv"])

train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sub_path, encoding="utf-8-sig")
w_train = pd.read_csv(w_train_path, encoding="utf-8-sig")
w_test = pd.read_csv(w_test_path, encoding="utf-8-sig") if w_test_path else pd.DataFrame()

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 날씨 데이터 통합 (보정용)
def prep_w(df):
    if df.empty: return df
    c = df.columns
    d_col = [x for x in c if "일시" in x or "날짜" in x][0]
    t_col = [x for x in c if "기온" in x][0]
    r_col = [x for x in c if "강수량" in x][0]
    res = df[[d_col, t_col, r_col]].copy()
    res.columns = ["일자", "기온", "강수량"]
    res["일자"] = pd.to_datetime(res["일자"])
    res["기온"] = pd.to_numeric(res["기온"], errors="coerce").interpolate().ffill().bfill()
    res["강수량"] = pd.to_numeric(res["강수량"], errors="coerce").fillna(0)
    res["is_rain"] = (res["강수량"] > 0).astype(int)
    res["is_hot"] = (res["기온"] >= 28).astype(int)
    res["is_cold"] = (res["기온"] <= 5).astype(int)
    return res

weather_all = pd.concat([prep_w(w_train), prep_w(w_test)]).drop_duplicates("일자")
train = train.merge(weather_all, on="일자", how="left")
test = test.merge(weather_all, on="일자", how="left")

In [ ]:
# 피처 생성 (학습용 + 월급날 효과 추가)
def add_advanced_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)
    
    # [추가] 월급날 효과 (25일 전후 2일간 외식 선호)
    df["is_payday"] = df["일"].apply(lambda x: 1 if x in [24, 25, 26] else 0)
    
    # 연휴 정보
    prev_gap = df["일자"].diff().dt.days.fillna(1)
    next_gap = df["일자"].shift(-1).sub(df["일자"]).dt.days.fillna(1)
    df["holiday_after"] = (prev_gap >= 2).astype(int)
    df["holiday_before"] = (next_gap >= 2).astype(int)
    
    # 운영 정보
    df["days_to_month_end"] = ((df["일자"] + pd.offsets.MonthEnd(0)) - df["일자"]).dt.days
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    return df

train = add_advanced_features(train)
test = add_advanced_features(test)
train["중식참여율"] = train["중식계"] / train["식사가능자수"]
train["석식참여율"] = train["석식계"] / train["식사가능자수"]

In [ ]:
# 베이스 모델 학습
lunch_f = ["월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수","is_fri","days_to_month_end","is_payday"]
dinner_f = ["월","일","요일","식사가능자수","본사출장자수","log_overtime","is_wed","is_payday"]

xgb_lunch = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)
xgb_dinner = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)

xgb_lunch.fit(train[lunch_f], train["중식참여율"])
xgb_dinner.fit(train[dinner_f], train["석식참여율"])

train["lunch_res"] = train["중식참여율"] - xgb_lunch.predict(train[lunch_f])
train["dinner_res"] = train["석식참여율"] - xgb_dinner.predict(train[dinner_f])

In [ ]:
# [업그레이드] 메뉴 보정 스무딩 (M-estimator)
def get_smoothed_menu_adj(train_df, menu_col, res_col, m=10):
    # m은 스무딩 강도: 횟수가 m번보다 적으면 전체 평균(0)으로 수렴시킴
    keyword_adj = {}
    keywords = ["돈까스", "제육", "치킨", "카레", "비빔밥", "짜장", "탕수육", "볶음밥", "오리", "생선", "불고기", "나물"]
    
    global_mean = 0 # 잔차의 전체 평균은 기본적으로 0에 가까움
    
    for kw in keywords:
        mask = train_df[menu_col].str.contains(kw, na=False)
        count = mask.sum()
        if count > 0:
            group_mean = train_df.loc[mask, res_col].mean()
            # 스무딩 공식: (평균 * 횟수 + 전역평균 * m) / (횟수 + m)
            smoothed_val = (group_mean * count + global_mean * m) / (count + m)
            keyword_adj[kw] = smoothed_val * 0.4 # 보정 강도 조절
    return keyword_adj

l_menu_adj = get_smoothed_menu_adj(train, "중식메뉴", "lunch_res")
d_menu_adj = get_smoothed_menu_adj(train[train["석식계"]>0], "석식메뉴", "dinner_res")

def apply_menu_adj(menu_text, adj_map):
    adj = 0
    for kw, val in adj_map.items():
        if kw in str(menu_text): adj += val
    return np.clip(adj, -0.035, 0.035)

In [ ]:
# 최종 예측 및 세밀한 보정
p_l_ratio = xgb_lunch.predict(test[lunch_f])
p_d_ratio = xgb_dinner.predict(test[dinner_f])

l_menu_sig = test["중식메뉴"].apply(lambda x: apply_menu_adj(x, l_menu_adj)).values
d_menu_sig = test["석식메뉴"].apply(lambda x: apply_menu_adj(x, d_menu_adj)).values

# 날씨 시그널
w_l_sig = 0.010 * test["is_rain"] - 0.005 * test["is_hot"] + 0.003 * test["is_cold"]
w_d_sig = 0.003 * test["is_rain"] + 0.002 * test["is_cold"]

# 연휴 및 특수일 시그널 (석식 수/금 보정 강화)
h_l_sig = -0.005 * test["holiday_before"] + 0.003 * test["holiday_after"]
h_d_sig = -0.007 * test["holiday_before"] + 0.002 * test["holiday_after"]

# [추가] 수요일/금요일이면서 연휴 전날인 경우 석식 추가 하락 (운영 심리)
d_special_drop = ((test["is_wed"] | test["is_fri"]) & test["holiday_before"]).astype(float) * -0.01

final_l_ratio = p_l_ratio + l_menu_sig + w_l_sig + h_l_sig
final_d_ratio = p_d_ratio + d_menu_sig + w_d_sig + h_d_sig + d_special_drop

pred_l = final_l_ratio * test["식사가능자수"]
pred_d = final_d_ratio * test["식사가능자수"]

# 제출 파일 생성
submission = sample_submission.copy()
submission[submission.columns[1]] = np.clip(pred_l.values, 0, None)
submission[submission.columns[2]] = np.clip(pred_d.values, 0, None)

submission.to_csv("inspired_final_v2.csv", index=False, encoding="utf-8-sig")
print("저장 완료: inspired_final_v2.csv")